# 02 — Data Preparation (Bab 5)

**Tujuan:** menghasilkan `data/processed/{train,val,test}.csv` yang bersih, bebas leakage, dan
siap dilatih pada Minggu 3–4.

**Urutan kerja:** 5.1 cleaning → 5.2 feature engineering → **5.3 splitting** → 5.4 transformasi →
5.5 balancing → 5.6 ringkasan.

> Mengapa splitting (5.3) dieksekusi sebelum transformasi (5.4), padahal penomoran laporan
> transformasi dulu? Karena `TfidfVectorizer` dan `StandardScaler` **belajar statistik dari data**
> (kosakata, IDF, mean, std). Kalau di-fit sebelum split, statistik test ikut terpakai saat latih —
> itu **data leakage**. Di laporan, urutan ini dijelaskan sebagai "split dulu, fit kemudian".

In [1]:
# ============================================================
# Konfigurasi global — satu sumber kebenaran untuk path & seed
# ============================================================
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

SEED = 42
# Notebook diasumsikan dijalankan dari folder notebooks/ (Jupyter/Colab/nbconvert).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("RAW :", RAW)

ROOT: D:\SEMESTER 5\TUGAS KELOMPOK PAK NANANG\id-gov-hatespeech-classification
RAW : D:\SEMESTER 5\TUGAS KELOMPOK PAK NANANG\id-gov-hatespeech-classification\data\raw


In [2]:
kamus = pd.read_csv(RAW / "new_kamusalay.csv", encoding="latin1", header=None,
                    names=["slang", "formal"])
abusive = pd.read_csv(RAW / "abusive.csv", encoding="latin1")
df = pd.read_csv(RAW / "data.csv", encoding="latin1")
print(f"data mentah: {df.shape[0]:,} baris x {df.shape[1]} kolom")

data mentah: 13,169 baris x 13 kolom


## 5.1 Data Cleaning

### Apa yang dibersihkan dan mengapa
1. **Label konflik** — satu teks dengan dua label `HS` berbeda tidak bisa dipercaya; dibuang (28 baris, 0,2%).
2. **Duplikat teks exact** — teks identik di train dan test membuat skor uji palsu.
3. **Token anonim `USER`/`URL`/`RT`** — bukan informasi linguistik; `USER` dihitung terpisah sebagai fitur.
4. **Kode hex `\xNN`** — artefak encoding, bukan kata.
5. **Simbol & tanda baca** — TF-IDF tidak membutuhkannya; jumlah `!`/`?` sudah jadi fitur terpisah.
6. **Slang/typo** — dinormalisasi dengan `new_kamusalay.csv` agar kata yang sama tidak terpecah jadi banyak varian.
7. **Stopword** — hanya dibuang pada versi *lexical*; kata negasi (`tidak`, `bukan`, `jangan`, `belum`, `tak`, `tiada`, `tanpa`)
   **selalu dipertahankan** karena membalik makna kalimat.

In [3]:
n_awal = len(df)

# (1) buang tweet yang label HS-nya bertentangan
label_per_tweet = df.groupby("Tweet")["HS"].nunique()
konflik = label_per_tweet[label_per_tweet > 1].index
df = df[~df["Tweet"].isin(konflik)].copy()
n_konflik = n_awal - len(df)

# (2) deduplikasi teks exact
sebelum_dedup = len(df)
df = df.drop_duplicates(subset=["Tweet"]).reset_index(drop=True)
n_dup = sebelum_dedup - len(df)

print(f"baris mentah                     : {n_awal:,}")
print(f"- label HS konflik (dibuang)     : {n_konflik:,}")
print(f"- duplikat teks exact (dibuang)  : {n_dup:,}")
print(f"= baris bersih                   : {len(df):,}")

baris mentah                     : 13,169
- label HS konflik (dibuang)     : 28
- duplikat teks exact (dibuang)  : 126
= baris bersih                   : 13,015


In [4]:
SLANG = dict(zip(kamus["slang"].str.strip(), kamus["formal"].str.strip()))
ABUSIVE = set(abusive["ABUSIVE"].str.lower().str.strip())

RE_ENTITY = re.compile(r"\b(?:user|url|rt)\b", flags=re.I)
RE_HEX = re.compile(r"\\x[0-9a-fA-F]{2}")
RE_SIMBOL = re.compile(r"[^A-Za-z\s]+")
RE_SPASI = re.compile(r"\s+")

def basic_clean(text: str) -> str:
    """Buang token anonim/hex/simbol, lowercase. Stopword BELUM dibuang."""
    t = RE_ENTITY.sub(" ", str(text))
    t = RE_HEX.sub(" ", t)
    t = RE_SIMBOL.sub(" ", t)
    return RE_SPASI.sub(" ", t).strip().lower()

def normalize_slang(text: str) -> str:
    """Ganti kata tidak baku per token; hasil mapping bisa berisi >1 kata."""
    return " ".join(SLANG.get(word, word) for word in text.split())

df["text_clean_contextual"] = df["Tweet"].map(basic_clean).map(normalize_slang)
df[["Tweet", "text_clean_contextual"]].head(4)

,Tweet,text_clean_contextual
0,- disaat semua cowok berusaha melacak perhatian gue. loe lantas remehkan perhatian yg gue kasih khusus ke elo. basic elo cowok bego ! ! !',di saat semua cowok berusaha melacak perhatian gue kamu lantas remehkan perhatian yang gue kasih khusus ke kamu basic kamu cowok bego
1,RT USER: USER siapa yang telat ngasih tau elu?edan sarap gue bergaul dengan cigax jifla calis sama siapa noh licew juga',siapa yang telat memberi tau kamu edan sarap gue bergaul dengan cigax jifla calis sama siapa itu licew juga
2,"41. Kadang aku berfikir, kenapa aku tetap percaya pada Tuhan padahal aku selalu jatuh berkali-kali. Kadang aku merasa Tuhan itu ninggalk...",kadang aku berpikir kenapa aku tetap percaya pada tuhan padahal aku selalu jatuh berkali kali kadang aku merasa tuhan itu meninggalkan a...
3,USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT TAPI DILIAT DARI MANA ITU AKU',aku itu aku dan ku tau matamu sipit tapi dilihat dari mana itu aku


In [5]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
NEGASI = {"tidak", "bukan", "jangan", "belum", "tak", "tiada", "tanpa"}
LEXICAL_STOPWORDS = STOPWORDS - NEGASI

df["text_clean_lexical"] = df["text_clean_contextual"].map(
    lambda t: " ".join(w for w in t.split() if w not in LEXICAL_STOPWORDS)
)

kosong = df["text_clean_contextual"].str.strip().eq("")
print(f"stopword Sastrawi: {len(STOPWORDS):,} kata; dipakai di lexical: {len(LEXICAL_STOPWORDS):,}")
print(f"dokumen kosong setelah cleaning: {int(kosong.sum())}")
df = df[~kosong].reset_index(drop=True)

df[["text_clean_contextual", "text_clean_lexical"]].head(4)

stopword Sastrawi: 809 kata; dipakai di lexical: 803
dokumen kosong setelah cleaning: 7


,text_clean_contextual,text_clean_lexical
0,di saat semua cowok berusaha melacak perhatian gue kamu lantas remehkan perhatian yang gue kasih khusus ke kamu basic kamu cowok bego,cowok berusaha melacak perhatian gue lantas remehkan perhatian gue kasih basic cowok bego
1,siapa yang telat memberi tau kamu edan sarap gue bergaul dengan cigax jifla calis sama siapa itu licew juga,telat tau edan sarap gue bergaul cigax jifla calis licew
2,kadang aku berpikir kenapa aku tetap percaya pada tuhan padahal aku selalu jatuh berkali kali kadang aku merasa tuhan itu meninggalkan a...,kadang berpikir percaya tuhan jatuh berkali kadang tuhan meninggalkan tuaku berencana berpisah kakakku memilih kristen anak ter
3,aku itu aku dan ku tau matamu sipit tapi dilihat dari mana itu aku,ku tau matamu sipit


**Catatan penting:** 7 dokumen menjadi **kosong total** setelah cleaning (isinya hanya emoji/simbol)
dan dibuang, sehingga total baris turun dari 13.015 menjadi **13.008**. Membiarkannya berarti
memberi model dokumen tanpa sinyal, yang hanya menambah noise.

**Contoh sebelum → sesudah** (perhatikan: normalisasi slang bisa memecah satu token menjadi beberapa kata,
dan kata negasi tetap ada di versi lexical).

In [6]:
contoh = df[df["Tweet"].str.contains(r"\bUSER\b|\\x", regex=True)].head(3)
for _, row in contoh.iterrows():
    print("MENTAH   :", row["Tweet"][:160])
    print("CONTEXT  :", row["text_clean_contextual"][:160])
    print("LEXICAL  :", row["text_clean_lexical"][:160])
    print("-" * 100)

MENTAH   : RT USER: USER siapa yang telat ngasih tau elu?edan sarap gue bergaul dengan cigax jifla calis sama siapa noh licew juga'
CONTEXT  : siapa yang telat memberi tau kamu edan sarap gue bergaul dengan cigax jifla calis sama siapa itu licew juga
LEXICAL  : telat tau edan sarap gue bergaul cigax jifla calis licew
----------------------------------------------------------------------------------------------------
MENTAH   : USER USER AKU ITU AKU\n\nKU TAU MATAMU SIPIT TAPI DILIAT DARI MANA ITU AKU'
CONTEXT  : aku itu aku dan ku tau matamu sipit tapi dilihat dari mana itu aku
LEXICAL  : ku tau matamu sipit
----------------------------------------------------------------------------------------------------
MENTAH   : USER USER Kaum cebong kapir udah keliatan dongoknya dari awal tambah dongok lagi hahahah'
CONTEXT  : kaum cebong kafir sudah kelihatan dongoknya dari awal tambah dungu lagi haha
LEXICAL  : kaum cebong kafir dongoknya dungu haha
--------------------------------------------

## 5.2 Feature Engineering

Fitur numerik diekstrak dari **teks mentah**, bukan teks bersih. Alasannya:
- `caps_ratio` dan `exclamation_count` hilang begitu teks di-lowercase/dibersihkan.
- `abusive_lexicon_count` dihitung dari teks lowercase **tanpa** normalisasi kamus, karena normalisasi
  dapat mengubah kata kasar menjadi kata baku dan membuat fitur ini *undercount*.

`is_gov` **tidak** termasuk fitur model; hanya penanda audit.

In [7]:
raw = df["Tweet"].astype(str)

df["char_length"] = raw.str.len()
df["word_count"] = raw.str.split().str.len()
df["caps_ratio"] = raw.map(lambda s: sum(ch.isupper() for ch in s) / (len(s) + 1))
df["exclamation_count"] = raw.str.count(r"[!?]")
df["user_mention_count"] = raw.str.count(r"\bUSER\b")
df["abusive_lexicon_count"] = raw.map(
    lambda s: sum(word in ABUSIVE for word in re.findall(r"[a-z]+", s.lower()))
)

NUM_FEATURES = ["char_length", "word_count", "caps_ratio",
                "exclamation_count", "user_mention_count", "abusive_lexicon_count"]

df[NUM_FEATURES].describe().round(3)

,char_length,word_count,caps_ratio,exclamation_count,user_mention_count,abusive_lexicon_count
count,13008.000,13008.000,13008.000,13008.000,13008.000,13008.000
mean,114.097,17.254,0.128,0.519,1.176,0.677
std,69.129,10.246,0.148,1.512,1.912,0.846
min,4.000,1.000,0.000,0.000,0.000,0.000
25%,59.000,9.000,0.043,0.000,0.000,0.000
50%,100.000,15.000,0.086,0.000,1.000,1.000
75%,152.000,23.000,0.154,0.000,1.000,1.000
max,561.000,52.000,0.882,43.000,24.000,15.000


In [8]:
GOV_KEYWORDS = [
    "presiden", "jokowi", "prabowo", "ahok", "anies", "pemerintah", "menteri",
    "dpr", "negara", "rakyat", "rezim", "partai", "politik", "kpk", "polri",
    "polisi", "korupsi", "pajak", "undang", "perppu", "gubernur", "pilkada", "nkri",
]
GOV_PATTERN = r"\b(?:" + "|".join(GOV_KEYWORDS) + r")\b"
df["is_gov"] = df["text_clean_contextual"].str.contains(GOV_PATTERN, regex=True).astype("int8")

print(f"tweet bertopik pemerintahan: {df['is_gov'].sum():,} ({df['is_gov'].mean():.2%})")
df.groupby("is_gov")[NUM_FEATURES].mean().round(3)

tweet bertopik pemerintahan: 4,147 (31.88%)


,char_length,word_count,caps_ratio,exclamation_count,user_mention_count,abusive_lexicon_count
is_gov,,,,,,
0,106.035,16.177,0.130,0.453,1.209,0.774
1,131.323,19.554,0.124,0.659,1.106,0.469


**Interpretasi fitur:** tweet bertopik pemerintahan rata-rata **lebih panjang** (131 vs 106 karakter),
punya lebih banyak tanda seru, tetapi **lebih sedikit** kata dari leksikon kasar (0,47 vs 0,77).
Artinya hate speech di ranah pemerintahan lebih sering berupa serangan/sindiran berbasis argumen
ketimbang makian mentah — dan justru inilah yang sulit bagi model leksikal. Temuan ini memperkuat
alasan memakai IndoBERT pada Minggu 5. `is_gov` tetap dikeluarkan dari `X` agar model tidak sekadar
belajar topik.

## 5.3 Splitting Data (70/15/15, stratified)

**Konsep:** data dibagi tiga kali secara acak **berlapis** (`stratify=y`) supaya proporsi kelas
di tiap split sama dengan populasi. `random_state=42` membuat pembagian ini bisa direproduksi
persis oleh siapa pun yang menjalankan notebook.

**Mengapa tiga split, bukan dua?** Validation dipakai untuk memilih hyperparameter dan
*early stopping* (Minggu 3–4). Test hanya dipakai **sekali** di Minggu 6, agar tidak ikut
"mempengaruhi" keputusan modeling = tidak ada *test set leakage*.

In [9]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["HS"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["HS"], random_state=SEED
)

split_info = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "jumlah_baris": [len(train_df), len(val_df), len(test_df)],
    "persen": [f"{len(s) / len(df):.1%}" for s in (train_df, val_df, test_df)],
    "HS=1": [int(s["HS"].sum()) for s in (train_df, val_df, test_df)],
    "persen_HS": [f"{s['HS'].mean():.2%}" for s in (train_df, val_df, test_df)],
})
split_info

,split,jumlah_baris,persen,HS=1,persen_HS
0,train,9105,70.0%,3848,42.26%
1,validation,1951,15.0%,824,42.23%
2,test,1952,15.0%,825,42.26%


In [10]:
# --- bukti tidak ada kebocoran antar-split ---
idx_train, idx_val, idx_test = (set(s.index) for s in (train_df, val_df, test_df))
assert idx_train.isdisjoint(idx_val) and idx_train.isdisjoint(idx_test)
assert idx_val.isdisjoint(idx_test)

teks_train = set(train_df["Tweet"])
assert not (teks_train & (set(val_df["Tweet"]) | set(test_df["Tweet"]))), \
    "Ada teks identik di train dan val/test -> leakage!"

assert len(train_df) + len(val_df) + len(test_df) == len(df)
print("OK: index antar-split disjoint, tidak ada teks duplikat lintas split.")
print(f"Total: {len(train_df):,} + {len(val_df):,} + {len(test_df):,} = {len(df):,} baris")

OK: index antar-split disjoint, tidak ada teks duplikat lintas split.
Total: 9,105 + 1,951 + 1,952 = 13,008 baris


## 5.4 Transformasi Data

Tiga jalur transformasi karena tiga model membutuhkan representasi berbeda:

| Jenis data | Transformasi | Dipakai model |
| :--- | :--- | :--- |
| Teks lexical | **TF-IDF** unigram–bigram, `max_features=5000`, `min_df=2` | Logistic Regression (Minggu 3) |
| Teks lexical | **TruncatedSVD** 100 komponen dari TF-IDF | XGBoost (Minggu 4) |
| Numerik | **StandardScaler** (fitur gaya penulisan) | XGBoost, bila diperlukan |
| Teks contextual | **Tokenizer IndoBERT** (`max_length=128`) | IndoBERT (Minggu 5) — dijalankan di notebook model |

Aturan emas: **fit hanya pada train**, lalu `transform` val/test. Blok di bawah membuktikannya
dengan `assert`, bukan sekadar klaim.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df["text_clean_lexical"])
X_val_tfidf = tfidf.transform(val_df["text_clean_lexical"])
X_test_tfidf = tfidf.transform(test_df["text_clean_lexical"])

assert X_train_tfidf.shape[1] == len(tfidf.vocabulary_) <= 5000
assert X_train_tfidf.shape[0] == len(train_df)
print(f"TF-IDF train : {X_train_tfidf.shape}  (dokumen x fitur)")
print(f"TF-IDF val   : {X_val_tfidf.shape}")
print(f"TF-IDF test  : {X_test_tfidf.shape}")
print(f"Contoh fitur : {list(tfidf.get_feature_names_out()[:8])}")

TF-IDF train : (9105, 5000)  (dokumen x fitur)
TF-IDF val   : (1951, 5000)
TF-IDF test  : (1952, 5000)
Contoh fitur : ['ab', 'abad', 'abadi', 'abal', 'abang', 'abdul', 'abdullah', 'abdullah bin']


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(train_df[NUM_FEATURES])

# Bukti 1: statistik scaler == statistik train, bukan statistik seluruh data
assert np.allclose(scaler.mean_, train_df[NUM_FEATURES].mean().to_numpy())
assert np.allclose(scaler.scale_, train_df[NUM_FEATURES].std(ddof=0).to_numpy())

X_train_num = scaler.transform(train_df[NUM_FEATURES])
X_val_num = scaler.transform(val_df[NUM_FEATURES])
X_test_num = scaler.transform(test_df[NUM_FEATURES])

# Bukti 2: hasil transform train tepat berpusat di 0
assert np.allclose(X_train_num.mean(axis=0), 0, atol=1e-9)

print(pd.DataFrame({
    "mean_train (fit)": scaler.mean_.round(4),
    "mean_seluruh_data (SALAH)": df[NUM_FEATURES].mean().round(4),
}, index=NUM_FEATURES).to_string())
print("\nOK: scaler hanya belajar dari train; val/test hanya di-transform.")

                       mean_train (fit)  mean_seluruh_data (SALAH)
char_length                    113.8134                   114.0969
word_count                      17.1979                    17.2538
caps_ratio                       0.1291                     0.1284
exclamation_count                0.5094                     0.5188
user_mention_count               1.1814                     1.1764
abusive_lexicon_count            0.6758                     0.6768

OK: scaler hanya belajar dari train; val/test hanya di-transform.


**Kenapa ini penting:** kolom "SALAH" menunjukkan nilai yang akan dipakai jika scaler di-fit pada
seluruh data. Selisihnya mungkin kecil di dataset ini, tetapi prinsipnya tetap: informasi test
tidak boleh masuk ke proses latih, sekalipun hanya berupa mean/std. Kebocoran kecil seperti ini
membuat estimasi performa terlalu optimistis dan tidak jujur saat dilaporkan.

In [13]:
# --- simpan hasil split; test set disegel sampai Minggu 6 ---
train_df.to_csv(PROCESSED / "train.csv", index=False)
val_df.to_csv(PROCESSED / "val.csv", index=False)
test_df.to_csv(PROCESSED / "test.csv", index=False)

import hashlib, json
manifest = {
    "seed": SEED,
    "sumber": "data/raw/data.csv",
    "baris_mentah": n_awal,
    "dibuang_label_konflik": n_konflik,
    "dibuang_duplikat": n_dup,
    "split": {k: len(v) for k, v in
              {"train": train_df, "val": val_df, "test": test_df}.items()},
    "sha256_teks_test": hashlib.sha256(
        "\n".join(test_df["Tweet"]).encode("utf-8")).hexdigest(),
}
(PROCESSED / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

{
  "seed": 42,
  "sumber": "data/raw/data.csv",
  "baris_mentah": 13169,
  "dibuang_label_konflik": 28,
  "dibuang_duplikat": 126,
  "split": {
    "train": 9105,
    "val": 1951,
    "test": 1952
  },
  "sha256_teks_test": "def612780eaea633c73575375dcf828330a95a01b9d2146d279b26add209b06e"
}


In [14]:
# --- verifikasi file tersimpan: dibaca ulang, bukan sekadar diasumsikan ---
train_cek = pd.read_csv(PROCESSED / "train.csv")
val_cek = pd.read_csv(PROCESSED / "val.csv")
test_cek = pd.read_csv(PROCESSED / "test.csv")

assert len(train_cek) == len(train_df)
assert len(val_cek) == len(val_df)
assert len(test_cek) == len(test_df)
assert {"Tweet", "HS", "text_clean_contextual", "text_clean_lexical", *NUM_FEATURES}.issubset(train_cek.columns)
print("OK: 3 file tersimpan dan terbaca ulang dengan jumlah baris yang sama.")
print(f"Kolom tersimpan ({train_cek.shape[1]}): {list(train_cek.columns)}")

OK: 3 file tersimpan dan terbaca ulang dengan jumlah baris yang sama.
Kolom tersimpan (22): ['Tweet', 'HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong', 'text_clean_contextual', 'text_clean_lexical', 'char_length', 'word_count', 'caps_ratio', 'exclamation_count', 'user_mention_count', 'abusive_lexicon_count', 'is_gov']


## 5.5 Data Balancing

**Prinsip:** balancing hanya boleh menyentuh **train**. Jika val/test di-resample, distribusi
kelas berubah dan evaluasi tidak lagi mencerminkan data nyata.

Keputusan proyek: **tidak melakukan resampling**. Alasan:
1. Rasio 42:58 masih ringan; model tidak "didominasi" kelas mayoritas sampai butuh koreksi paksa.
2. `class_weight='balanced'` (spesifikasi `CONTEXT.md`) sudah menangani ketimpangan pada fungsi kerugian
   tanpa mengubah data dan tanpa menggandakan dokumen.
3. Resampling teks (oversampling/SMOTE) menciptakan duplikat atau teks sintetis yang tidak wajar —
   risiko overfitting dan evaluasi yang menyesatkan.

In [15]:
from sklearn.utils.class_weight import compute_class_weight

hitung = train_df["HS"].value_counts().sort_index()
rasio = hitung.max() / hitung.min()
bobot = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["HS"])

print(f"Distribusi train: HS=0 {hitung[0]:,} | HS=1 {hitung[1]:,} | rasio {rasio:.2f}:1")
print("Bobot kelas (dihitung dari TRAIN saja):",
      {0: round(bobot[0], 4), 1: round(bobot[1], 4)})
print("Dipakai sebagai: LogisticRegression(class_weight='balanced') / class_weight=dict pada XGBoost.")
print("Data train dan val/test TIDAK diubah jumlahnya.")

Distribusi train: HS=0 5,257 | HS=1 3,848 | rasio 1.37:1
Bobot kelas (dihitung dari TRAIN saja): {0: np.float64(0.866), 1: np.float64(1.1831)}
Dipakai sebagai: LogisticRegression(class_weight='balanced') / class_weight=dict pada XGBoost.
Data train dan val/test TIDAK diubah jumlahnya.


## 5.6 Ringkasan Apa–Mengapa–Bagaimana

| Langkah | Apa yang dilakukan | Mengapa | Bagaimana |
| :--- | :--- | :--- | :--- |
| Deduplikasi | Buang 126 duplikat teks exact (dari 146 baris duplikat; 20 lainnya sudah ikut terbuang bersama label konflik) | Teks identik di train & test = leakage, skor uji palsu | `drop_duplicates(subset='Tweet')` sebelum split |
| Label konflik | Buang 28 baris (8 tweet) berlabel `HS` ganda | Label tidak reliabel, mengajarkan noise | `groupby('Tweet')['HS'].nunique() > 1` |
| Dokumen kosong | Buang 7 dokumen yang habis setelah cleaning | Tidak ada sinyal teks yang bisa dipelajari | filter `text_clean_contextual == ''` |
| Anonim & simbol | Buang `USER`/`URL`/`RT`, hex, tanda baca | Bukan sinyal linguistik; `USER` dihitung sebagai fitur | regex bertahap |
| Normalisasi slang | Kata tidak baku → baku via `new_kamusalay.csv` | Mengurangi variasi ejaan yang memecah sinyal | mapping token-per-token (O(1) dict) |
| Stopword | Buang stopword di versi lexical, **kecuali negasi** | Kurangi dimensi TF-IDF; negasi mengubah makna | Sastrawi dikurangi himpunan negasi |
| Dua varian teks | `contextual` (utuh) & `lexical` (tanpa stopword) | Kebutuhan IndoBERT ≠ TF-IDF | dua kolom hasil cleaning |
| Fitur gaya | 6 fitur numerik dari teks mentah | Menangkap emosi/gaya yang hilang saat cleaning | regex & rasio sederhana |
| Fitur kasar | `abusive_lexicon_count` dari lowercase pra-normalisasi | Normalisasi bisa menghapus kata kasar | lookup `abusive.csv` |
| Split | 70/15/15 stratified, `random_state=42` | Evaluasi adil & reproducible; val untuk tuning, test disegel | `train_test_split(stratify=y)` 2 tahap |
| TF-IDF | unigram–bigram, 5.000 fitur, fit di train | Representasi teks untuk model linier | `fit_transform(train)`, `transform(val/test)` |
| Scaling | `StandardScaler` fit di train | Samakan skala fitur numerik | `fit(train)`, `transform(val/test)` |
| Balancing | Tanpa resampling, pakai `class_weight` | Imbalance ringan; hindari data sintetis/duplikat | `compute_class_weight('balanced')` |
| Test set | Disimpan, tidak disentuh sampai Minggu 6 | Menjaga estimasi performa akhir tetap jujur | `test.csv` + `split_manifest.json` (hash teks) |

**Jumlah sampel final**

| Split | Baris | HS=1 | Proporsi HS |
| :--- | ---: | ---: | ---: |
| Train | 9.105 | 3.848 | 42,26% |
| Validation | 1.951 | 824 | 42,23% |
| Test | 1.952 | 825 | 42,26% |
| **Total** | **13.008** | **5.497** | **42,26%** |

> Angka pada tabel di atas berasal dari split `random_state=42` atas data setelah deduplikasi.
> Bila dataset mentah diganti, jalankan ulang notebook dan perbarui tabel ini.